# 06 - Evaluation, LLM-as-a-Judge, and Error Analysis

This notebook performs objective and judge-based evaluation with per-task latency and retrieval diagnostics.


## What Is This Technique?

### What is this technique?
Multi-axis query generation evaluation

### Definition and core concepts.
A unified evaluation framework combining exact/syntax/grounding scores, text metrics, retrieval diagnostics, latency, and judge assessments.

### Why was this technique developed?
Single metrics can hide critical failures (for example, fluent but non-executable queries).

### What limitations of traditional RAG does it solve?
Traditional RAG evaluation focuses answer relevance/faithfulness. Here we require executable query correctness and schema alignment.

### Architecture and workflow diagram explanation.
Load predictions -> compute metrics -> aggregate by task/model -> plot distributions -> run judge analysis.

### Component-by-component breakdown.
SQL/Cypher parsers, overlap metrics, schema-grounding checks, retrieval recall metrics, latency aggregators, plotting utilities.

### When should it be used in real-world systems?
Use for release gating, regression testing, and deciding whether fine-tuning quality justifies operational cost.

### Advantages and disadvantages.
Advantages: high diagnostic coverage. Disadvantages: more compute and longer evaluation cycles.

### Comparison against standard RAG and other implemented RAG variants.
Compared with standard RAG metrics, this evaluation explicitly measures executable query behavior and schema retrieval quality.

### Implementation details and design decisions used in this project.
Evaluation now reports retrieval table recall and latency p50/p95 alongside classic SQL/Cypher metrics.


In [ ]:
from repo_query_gen.inference import run_batch_inference

# Build inference artifacts for evaluation.
run_batch_inference("fast", "baseline_granite", "baseline_granite")
run_batch_inference("fast", "baseline_qwen", "baseline_qwen")
try:
    run_batch_inference("fast", "finetuned", "finetuned")
except FileNotFoundError:
    print("Finetuned adapter missing, continuing baseline-only.")


In [ ]:
from repo_query_gen.evaluation import run_evaluation_bundle

artifacts = run_evaluation_bundle("fast")
artifacts


In [ ]:
import json
import pandas as pd
from pathlib import Path

eval_dir = Path("../artifacts/evaluation/fast")
metrics = pd.read_csv(eval_dir / "metrics_per_example.csv")
print(metrics.groupby(["label", "task"])[["exact_match", "syntax_success", "schema_grounding", "retrieval_table_recall", "generation_latency_ms"]].mean())

for summary_path in sorted(eval_dir.glob("summary_*.json")):
    print(summary_path.name)
    print(json.loads(summary_path.read_text()))


## Post-Run Analysis (evaluation and metric interpretation)

After executing the real pipeline, use this section to document measured outcomes.

- Analyze actual outputs, metrics, retrieval quality, latency, and generation quality.
- Explain how the technique changed measured behavior vs baseline.
- Interpret every metric in business and engineering terms.
- Capture failure modes, lessons learned, and practical takeaways.
- Explain why specific outputs were produced and what they demonstrate.
- Conclude effectiveness based on measured evidence, not assumptions.
